# Inclusive-jet data distributions

Beam-orientation comparisons for configurable reconstructed-jet selections. For the unflipped-Lab, flipped-Lab, CM, and raw-pT/unflipped-Lab views, plot common-scale Pb-going, p-going, and combined 2D maps, normalized pT spectra with ratios to combined, and eta projections with ratios to combined.


In [ ]:
%load_ext autoreload
%autoreload 2
from dataclasses import replace
from pathlib import Path
import os, sys
PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None: raise RuntimeError('Run this notebook from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'), Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path: sys.path.insert(0, str(path))
    import ROOT
ROOT.gROOT.SetBatch(True); ROOT.gStyle.SetOptStat(0); ROOT.TH1.AddDirectory(False)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import SINGLE_JET_PT_BINS
from hist_analysis.python.data_distributions import (
    draw_orientation_comparisons,
)
from hist_analysis.python.root_style import DEFAULT_PLOT_STYLE


In [ ]:
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
SELECTION = 'jetId'  # jetId, trkMax, or noSel
if SELECTION not in {'jetId', 'trkMax', 'noSel'}: raise ValueError(f'Unsupported SELECTION={SELECTION!r}')
OUTPUT_DIR = Path(os.environ.get('DATA_JET_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis/output/data_jet_distributions')) / SELECTION
TRIGGERS = ('MinimumBias', 'Jet60', 'Jet80', 'Jet100')
STEMS = {'MinimumBias': 'MB', 'Jet60': 'Jet60', 'Jet80': 'Jet80', 'Jet100': 'Jet100'}
COMBINED_FILES = {
    'MinimumBias': DATA_DIR / f'mb_ak4_{SELECTION}.root', 'Jet60': DATA_DIR / f'jet60_ak4_{SELECTION}.root',
    'Jet80': DATA_DIR / f'jet80_ak4_{SELECTION}.root', 'Jet100': DATA_DIR / f'jet100_ak4_{SELECTION}.root',
}
DIRECTION_FILES = {trigger: {
    'Pb-going': DATA_DIR / 'Pbgoing' / f'{STEMS[trigger]}_Pbgoing_ak4_{SELECTION}.root',
    'p-going': DATA_DIR / 'pgoing' / f'{STEMS[trigger]}_pgoing_ak4_{SELECTION}.root',
    'combined': COMBINED_FILES[trigger],
} for trigger in TRIGGERS}
FRAMES = {
    'lab_unflipped': ('Lab unflipped', 'hRecoInclusiveJetPtEtaLabUnflipped', '#eta_{Lab,unflipped}^{jet}'),
    'raw_lab_unflipped': ('Raw pT, Lab unflipped', 'hRecoInclusiveJetRawPtEtaLabUnflipped', '#eta_{Lab,unflipped}^{jet}', 'p_{T}^{raw,jet}'),
    'lab': ('Lab flipped', 'hRecoInclusiveJetPtEtaLab', '#eta_{Lab}^{jet}'),
    'cm': ('CM', 'hRecoInclusiveJetPtEtaCM', '#eta_{CM}^{jet}'),
}
JET_PT_BINS = {trigger: list(SINGLE_JET_PT_BINS) for trigger in TRIGGERS}
JET_PT_BINS['Jet80'] = [
    (40, 50), (50, 60), (60, 70), (70, 80), (80, 90),
    (90, 100), (100, 110), (110, 120), (120, 130),
]
PT_ETA_RANGE = (-2.5, 2.5)  # half-open eta range for pT projections
PT_ORIENTATION_NORMALIZATION_RANGE = (110.0, 130.0)
REBIN_PT = 1; REBIN_ETA = 1; PT_DISPLAY_RANGE = (40.0, 500.0)
ORIENTATION_RATIO_RANGE = (0.75, 1.25)
SAVE_PNG = False; DRAW_GRID = True
PLOT_STYLE = replace(DEFAULT_PLOT_STYLE, annotation_text_size=0.028, legend_text_size=0.028)
missing = [str(path) for path in COMBINED_FILES.values() if not path.exists()]
missing += [str(path) for files in DIRECTION_FILES.values() for path in files.values() if not path.exists()]
if missing: raise FileNotFoundError('Missing ROOT files:\n' + '\n'.join(sorted(set(missing))))


## Pb-going and p-going overlays and ratios to combined

In [ ]:
mb_orientation_results = draw_orientation_comparisons(
    'MinimumBias', DIRECTION_FILES['MinimumBias'], FRAMES, JET_PT_BINS['MinimumBias'],
    jet_kind='jet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=PT_ETA_RANGE,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE,
)

In [ ]:
jet60_orientation_results = draw_orientation_comparisons(
    'Jet60', DIRECTION_FILES['Jet60'], FRAMES, JET_PT_BINS['Jet60'],
    jet_kind='jet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=PT_ETA_RANGE,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE,
)

In [ ]:
jet80_orientation_results = draw_orientation_comparisons(
    'Jet80', DIRECTION_FILES['Jet80'], FRAMES, JET_PT_BINS['Jet80'],
    jet_kind='jet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=PT_ETA_RANGE,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE,
)

In [ ]:
jet100_orientation_results = draw_orientation_comparisons(
    'Jet100', DIRECTION_FILES['Jet100'], FRAMES, JET_PT_BINS['Jet100'],
    jet_kind='jet', output_dir=OUTPUT_DIR, rebin_eta=REBIN_ETA, rebin_pt=REBIN_PT,
    pt_display_range=PT_DISPLAY_RANGE, pt_eta_range=PT_ETA_RANGE,
    pt_normalization_range=PT_ORIENTATION_NORMALIZATION_RANGE,
    selection=SELECTION,
    ratio_range=ORIENTATION_RATIO_RANGE, save_png=SAVE_PNG,
    grid=DRAW_GRID, style=PLOT_STYLE,
)